# SIS3: Generative Model for Text Generation
## Part 1: Baseline Implementation — Character-Level GRU (PyTorch)

## 1. Install & Import Dependencies

In [ ]:
import os
import math
import time
import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Load Dataset

In [ ]:
DATA_PATH = 'input.txt'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'Sample text:\n{text[:200]}')

Total characters: 1,115,394
Sample text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 3. Vocabulary & Encoding

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')

char2idx = {ch: idx for idx, ch in enumerate(chars)}
idx2char = {idx: ch for ch, idx in char2idx.items()}

encoded = np.array([char2idx[ch] for ch in text], dtype=np.int64)
print(f'Encoded length: {len(encoded):,}')

Vocabulary size: 65
Encoded length: 1,115,394


## 4. Dataset & DataLoader

In [ ]:
# Baseline Model Hyperparameters (from Table 1)
SEQ_LENGTH    = 100
EMBED_DIM     = 256
LSTM_UNITS    = 512
DROPOUT_RATE  = 0.2
BATCH_SIZE    = 512
LEARNING_RATE = 0.001
NUM_EPOCHS    = 5
VAL_SPLIT     = 0.2

class ShakespeareDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx: idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + self.seq_len], dtype=torch.long)
        return x, y

# Train/validation split
split_idx = int(len(encoded) * (1 - VAL_SPLIT))
train_data = encoded[:split_idx]
val_data   = encoded[split_idx:]

train_dataset = ShakespeareDataset(train_data, SEQ_LENGTH)
val_dataset   = ShakespeareDataset(val_data,   SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2, pin_memory=True)

print(f'Train sequences: {len(train_dataset):,}')
print(f'Val   sequences: {len(val_dataset):,}')

Train sequences: 892,215
Val   sequences: 222,979


## 5. Model Architecture

In [ ]:
class CharGRU(nn.Module):
    """
    Baseline character-level GRU:
      Embedding(vocab_size, embed_dim)
      -> GRU(embed_dim, lstm_units, num_layers=2)
      -> Dropout(dropout_rate)
      -> Linear(lstm_units, vocab_size)
    """
    def __init__(self, vocab_size, embed_dim, lstm_units, dropout_rate, num_layers=2):
        super(CharGRU, self).__init__()
        self.lstm_units = lstm_units
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=lstm_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(lstm_units, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)               # (batch, seq, embed_dim)
        out, hidden = self.gru(x, hidden)   # (batch, seq, lstm_units)
        out = out[:, -1, :]                 # take last timestep
        out = self.dropout(out)
        logits = self.fc(out)               # (batch, vocab_size)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        h0 = torch.zeros(self.num_layers, batch_size, self.lstm_units).to(device)
        return h0  # GRU has only hidden state (no cell state)


model = CharGRU(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    lstm_units=LSTM_UNITS,
    dropout_rate=DROPOUT_RATE
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

CharGRU(
  (embedding): Embedding(65, 256)
  (gru): GRU(256, 512, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=512, out_features=65, bias=True)
)

Total trainable parameters: 2,808,641


## 6. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses   = []

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            if training:
                optimizer.zero_grad()
            logits, _ = model(x_batch)
            loss = criterion(logits, y_batch)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

print('Starting training...\n')
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, training=True)
    val_loss   = run_epoch(val_loader,   training=False)
    elapsed = time.time() - t0

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_ppl = math.exp(train_loss)
    val_ppl   = math.exp(val_loss)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} (PPL {train_ppl:.2f}) | '
          f'Val Loss: {val_loss:.4f} (PPL {val_ppl:.2f}) | '
          f'Time: {elapsed:.1f}s')

total_time = time.time() - start_time
print(f'\nTotal training time: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Final Train Loss: {train_losses[-1]:.4f}  |  Final Val Loss: {val_losses[-1]:.4f}')
print(f'Final Train Perplexity: {math.exp(train_losses[-1]):.2f}  |  Final Val Perplexity: {math.exp(val_losses[-1]):.2f}')

Starting training...



c:\python313\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 7. Loss Curves

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, train_losses, label='Train Loss', marker='o')
axes[0].plot(epochs_range, val_losses,   label='Val Loss',   marker='s')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True)

train_ppl_list = [math.exp(l) for l in train_losses]
val_ppl_list   = [math.exp(l) for l in val_losses]
axes[1].plot(epochs_range, train_ppl_list, label='Train Perplexity', marker='o', color='orange')
axes[1].plot(epochs_range, val_ppl_list,   label='Val Perplexity',   marker='s', color='red')
axes[1].set_title('Training & Validation Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('loss_curves_gru.png', dpi=150)
plt.show()
print('Plot saved.')

## 8. Save Model Checkpoint

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'char2idx': char2idx,
    'idx2char': idx2char,
    'vocab_size': vocab_size,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'hyperparams': {
        'SEQ_LENGTH': SEQ_LENGTH, 'EMBED_DIM': EMBED_DIM,
        'LSTM_UNITS': LSTM_UNITS, 'DROPOUT_RATE': DROPOUT_RATE,
        'BATCH_SIZE': BATCH_SIZE, 'LEARNING_RATE': LEARNING_RATE,
    }
}, 'baseline_gru_checkpoint.pt')
print('Checkpoint saved to baseline_gru_checkpoint.pt')

## 9. Text Generation

In [ ]:
def generate_text(model, seed_text, char2idx, idx2char, length=500, temperature=1.0, device='cpu'):
    """
    Generate text from the trained model.

    Args:
        seed_text  : Starting string (at least SEQ_LENGTH chars recommended)
        length     : Number of characters to generate
        temperature: Sampling temperature (higher = more random, lower = more greedy)
    """
    model.eval()
    generated = seed_text

    # Encode seed
    input_seq = [char2idx.get(ch, 0) for ch in seed_text[-SEQ_LENGTH:]]

    hidden = model.init_hidden(1, device)

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_seq[-SEQ_LENGTH:]], dtype=torch.long).to(device)
            logits, hidden = model(x, hidden)

            # Apply temperature scaling
            logits = logits / temperature
            probs  = torch.softmax(logits, dim=-1).squeeze()

            # Sample next character
            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_char = idx2char[next_idx]

            generated += next_char
            input_seq.append(next_idx)

    return generated


seed = "ROMEO:\nBut soft, what light through yonder window breaks?\nIt is the east, and Juliet is"

print('=' * 60)
print('Generated Text — Temperature 0.5 (more conservative):')
print('=' * 60)
print(generate_text(model, seed, char2idx, idx2char, length=500, temperature=0.5, device=device))

print('\n' + '=' * 60)
print('Generated Text — Temperature 1.0 (balanced):')
print('=' * 60)
print(generate_text(model, seed, char2idx, idx2char, length=500, temperature=1.0, device=device))

print('\n' + '=' * 60)
print('Generated Text — Temperature 1.5 (more creative):')
print('=' * 60)
print(generate_text(model, seed, char2idx, idx2char, length=500, temperature=1.5, device=device))

## 10. Performance Summary

| Metric | Value |
|---|---|
| Final Training Loss | *(recorded after training)* |
| Final Validation Loss | *(recorded after training)* |
| Final Training Perplexity | exp(train_loss) |
| Final Validation Perplexity | exp(val_loss) |
| Total Training Time | *(recorded above)* |
| Total Trainable Parameters | *(printed in Section 5)* |

> **Note:** Perplexity = exp(cross-entropy loss). Lower is better.